In [ ]:
# upload train data
!wget https://raw.githubusercontent.com/nyu-mll/CoLA-baselines/master/acceptability_corpus/raw/in_domain_train.tsv

In [6]:
import os
import pandas as pd
from sklearn.utils import shuffle

In [7]:
# preprocess raw data
df = pd.read_csv('in_domain_train.tsv', sep = '\t', header=None)

df = df.drop(columns=[2]).rename(columns={0: 'id', 1: 'label', 3: 'text'})
df['label'] = df['label'].replace({0: 'negative', 1: 'positive'})
df['id'] = df.index
df = df[['text', 'label', 'id']]

df.head()

,text,label,id
0,"Our friends won't buy this analysis, let alone...",positive,0
1,One more pseudo generalization and I'm giving up.,positive,1
2,One more pseudo generalization or I'm giving up.,positive,2
3,"The more we study verbs, the crazier they get.",positive,3
4,Day by day the facts are getting murkier.,positive,4


In [8]:
# define formatting functions
def format_row(row):
    question = f"question : Is this text grammatically correct?"
    answer = f"answer : {'yes' if row['label'] == 'positive' else 'no'}"
    context = f"context : {row['text']}"
    text = f"{question}\n{answer}\n{context}\n"
    return pd.Series([row['id'], text])

def create_balanced_sample(df, number_of_rows, label_value, random_state=13):
    sample_df_match = df[df['label'] == label_value].sample(
        n=(number_of_rows // 2), random_state=random_state)
    sample_df_unmatch = df[df['label'] != label_value].sample(
        n=(number_of_rows - number_of_rows // 2), random_state=random_state)
    sample_df = shuffle(pd.concat([sample_df_match, sample_df_unmatch],
                          axis=0, ignore_index=True))
    return sample_df

def create_sample_tables(random_state=13, number_of_rows=32):
    sample_df = create_balanced_sample(
        df, number_of_rows, 'positive', random_state)
    
    print(f"Number of positive rows: {(sample_df['label'] == 'positive').sum()}")
    print(f"Number of negative rows: {(sample_df['label'] == 'negative').sum()}")
    
    os.makedirs("data/cls/cola/train/", exist_ok=True)
    
    sample_df.to_csv(f'data/cls/cola/train/train_{random_state}.csv', index=False)
    sample_df_qac = sample_df.apply(format_row, axis=1)
    sample_df_qac = sample_df_qac.rename(columns={0: 'id', 1: 'text'})
    sample_df_qac.to_csv(f'data/cls/cola/train_qac_{random_state}.csv', index=False)

In [ ]:
# create train samples
# this code will create 2 files: 
# * train_{random_state}.csv in data/cls/cola/train folder
# * train_qac_{random_state}.csv in data/cls/cola folder
create_sample_tables(random_state=13, number_of_rows=200)

In [ ]:
!rm in_domain_train.tsv

----